In [ ]:
# Sequence similarity plots (ESM2)

# Analyzes the sequence comparison results of the ESM2 pipeline, i.e. the comparison of each generated protein against the reference toxins.

# The notebook reads the CSV file given in `CSV_FILENAME` (expected columns: `Generated_Protein_ID`, `Toxin_ID`, `Percent_Identity`, `Alignment_Score`) and
# - plots the distribution of the percent identity across all comparisons (histogram),
# - collects all hits with a percent identity above 20 % per generated protein and reference toxin (used to determine the toxin with the most hits),
# - creates violin plots of the percent identity (including the 30 % sequence similarity threshold) and of the alignment score for each reference toxin. 
# The toxin with the most hits is shown first and highlighted in pink.

# All plots are saved as PDF files in `PLOTS_DIR`.

In [ ]:
# ---------------------------------------------------------------------------
# Configuration — edit these values to match your setup.
# ---------------------------------------------------------------------------

# Directory containing the sequence comparison result CSV file.
BASE_DIR = "/path_to_your_base_directory/ESM2_filtered"

# Directory where the generated plots (PDFs) are written.
PLOTS_DIR = "/path_to_your_base_directory/ESM2"

# Input CSV with the sequence comparison results.
CSV_FILENAME = "sequence_comparison_1KBA_fixed_AS_filtered.csv"

# Output plot filenames.
HISTOGRAM_FILENAME = "histogram_1KBA_fixed_AS_sequence_compare.pdf"
VIOLIN_IDENTITY_FILENAME = "violinplot_1KBA_fixed_AS_sequence_identity.pdf"
VIOLIN_ALIGNMENT_FILENAME = "violinplot_1KBA_fixed_AS_sequence_similarity_alignmentscore.pdf"
# ---------------------------------------------------------------------------


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os

In [ ]:
# Global plot styling
plt.rcParams['axes.titlesize'] = 24
plt.rcParams['axes.labelsize'] = 22
plt.rcParams['xtick.labelsize'] = 20
plt.rcParams['ytick.labelsize'] = 20
plt.rcParams['legend.fontsize'] = 18
plt.rcParams['legend.title_fontsize'] = 18

In [ ]:
path_to_csv = os.path.join(BASE_DIR, CSV_FILENAME)
df_sequence_comparison = pd.read_csv(path_to_csv)
print(f"DataFrame loaded with {len(df_sequence_comparison)} rows and {len(df_sequence_comparison.columns)} columns.")

In [ ]:
def get_base_id(protein_id):
    """Strip AlphaFold suffixes (_unrelaxed, _relaxed, _seed_*) from a protein ID."""
    match = re.search(r'(.+?)(_unrelaxed|_relaxed)', protein_id)
    if match:
        return match.group(1).strip()
    match = re.search(r'(.+?)(_seed_)', protein_id)
    if match:
        return match.group(1).strip()
    return protein_id.strip()

df_sequence_comparison['Generated_Protein_ID_short'] = df_sequence_comparison['Generated_Protein_ID'].apply(get_base_id)

In [ ]:
# Histogram: distribution of percent identity across all comparisons
plt.figure(figsize=(10, 6))
sns.histplot(df_sequence_comparison['Percent_Identity'], bins=60, kde=False)
plt.title('Distribution of Percent Identity Across All Comparisons')
plt.xlabel('Percent Identity')
plt.ylabel('Frequency (Number of Comparisons)')
plt.savefig(os.path.join(PLOTS_DIR, HISTOGRAM_FILENAME), format="pdf")

In [ ]:
# Heatmap of top hits (Percent Identity > 20%)
df_filtered = df_sequence_comparison[df_sequence_comparison['Percent_Identity'] > 20.0].copy()
if not df_filtered.empty:
    df_filtered['Generated_Protein_ID_short'] = df_filtered['Generated_Protein_ID_short'].str.strip()
    df_filtered['Toxin_ID'] = df_filtered['Toxin_ID'].str.strip()
    print(f"Filtered DataFrame shape: {df_filtered.shape}")
    print(f"Unique protein IDs before aggregation: {df_filtered['Generated_Protein_ID_short'].nunique()}")

    aggregated_data = df_filtered.groupby(['Generated_Protein_ID_short', 'Toxin_ID'])['Percent_Identity'].mean().reset_index()
    pivot_table = aggregated_data.pivot_table(index='Generated_Protein_ID_short', columns='Toxin_ID', values='Percent_Identity')
    print(f"Final heatmap data shape: {pivot_table.shape}")
else:
    print("No matches above threshold found for the heatmap.")

In [ ]:
# Violin plot: percent identity by toxin
count_toxin_similarities = pivot_table.count()
toxin_with_highest_similarity = count_toxin_similarities.idxmax().replace(':A', '')

custom_ordered_toxins = ['3PLC','4OM5','2H8U','3VTS','6RC7','1UG4','1VYC','2H5F','1ANS','1QDP','1KBA','4LFT','1DTK','2ASC','2ATB','2I61','1FCU','4K09','1VIB','2MLT']
if toxin_with_highest_similarity in custom_ordered_toxins:
    custom_ordered_toxins.remove(toxin_with_highest_similarity)
final_order = [toxin_with_highest_similarity] + custom_ordered_toxins
custom_colors = ['pink'] + ['gray'] * len(custom_ordered_toxins)

df_plot_data = df_sequence_comparison.copy()
df_plot_data['Toxin_ID'] = df_plot_data['Toxin_ID'].str.split(':').str[0]

plt.figure(figsize=(17, 8))
sns.violinplot(data=df_plot_data, x='Toxin_ID', y='Percent_Identity', order=final_order, palette=custom_colors)
plt.title('Sequence similarity for all data by toxin\n')
plt.axhline(y=30, color='green', linestyle='--', label='Sequence similarity threshold')
plt.legend()
plt.xlabel('\nToxin ID')
plt.ylabel('Sequence Similarity in %\n')
plt.ylim(0, 100)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, VIOLIN_IDENTITY_FILENAME), format="pdf")

In [ ]:
# Violin plot: alignment score by toxin
plt.figure(figsize=(17, 8))
sns.violinplot(data=df_plot_data, x='Toxin_ID', y='Alignment_Score', order=final_order, palette=custom_colors)
plt.title('Sequence similarity (Alignment Score) for all data by toxin\n')
plt.legend()
plt.xlabel('\nToxin ID')
plt.ylabel('Alignment Score\n')
plt.ylim(-200, 300)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, VIOLIN_ALIGNMENT_FILENAME), format="pdf")